# Scratch notebook
Use this for experiments. Keep `starter.ipynb` clean.

In [ ]:
# # Install required libraries
# # Run this cell, then restart your notebook kernel if necessary.
# !pip install -q -U transformers accelerate peft trl datasets bitsandbytes torch


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# Cell 2: Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading base model onto GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # This automatically puts the model on your RunPod GPU
)

print(f"Model loaded successfully on: {model.device}")

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading base model onto GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'. Are this object's requirements defined correctly?

## Multi-stage IOED pipeline (T1 → T4)

We mirror the Rozenblit & Keil (2002) Illusion of Explanatory Depth protocol on Qwen for each item in `mechanism_qa.jsonl`.

| Stage | Probe | What it tests |
|---|---|---|
| **T1** | "How confident can you explain this mechanism?" *(no solving yet)* | Prior over own ability |
| **T2** | Solve + explain → ask post-explanation confidence | Does fluent self-output inflate confidence? |
| **T3** | Diagnostic probe using `rubric_notes` (the structural element shallow answers miss) → ask confidence | Does forced articulation collapse confidence (human IOED pattern)? |
| **T4** | Show `reference_answer` → ask confidence | Updates after seeing ground truth |

The chat history is **persistent across stages** — at T3 and T4 the model sees its own prior explanation, mirroring how humans in IOED studies are confronted with their own earlier output.

In [ ]:
# Helpers and dataset load
import json, re
from collections import Counter
from pathlib import Path

DATASET_PATH = Path("/workspace/ARK-Interpretability/notebooks/mechanism_qa.jsonl")
RESULTS_DIR  = Path("/workspace/ARK-Interpretability/results")

items = [json.loads(l) for l in DATASET_PATH.open()]
print(f"Loaded {len(items)} items")
print(f"  topic_buckets: {dict(Counter(i['topic_bucket'] for i in items))}")
print(f"  splits:        {dict(Counter(i['split'] for i in items))}")


def chat(messages, max_new_tokens=256, temperature=None, do_sample=True):
    """One assistant turn given message history. Returns the new assistant text."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tokenizer.eos_token_id)
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    outputs = model.generate(**inputs, **kwargs)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def parse_confidence(text):
    """Extract a percentage 0-100 from free-form text. Returns None if not found."""
    m = re.search(r"(\d{1,3}(?:\.\d+)?)\s*%?", text.strip())
    if not m: return None
    v = float(m.group(1))
    return v if 0 <= v <= 100 else None


def ask_confidence(history, prompt):
    """Append a confidence-eliciting prompt (greedy decode), parse the percentage."""
    msgs = history + [{
        "role": "user",
        "content": prompt + "\n\nReply with ONLY a single integer from 0 to 100. No words, no symbols.",
    }]
    raw = chat(msgs, max_new_tokens=10, do_sample=False)
    return parse_confidence(raw), raw


In [ ]:
def run_ioed_pipeline(item, verbose=False):
    """Run the T1 -> T4 IOED ladder on one mechanism_qa item."""
    question  = item["question"]
    rubric    = item["rubric_notes"]
    reference = "\n".join(f"{i+1}. {s}" for i, s in enumerate(item["reference_answer"]))

    history = [{"role": "system", "content": "You are a helpful and honest AI assistant."}]
    out = {
        "item_id":      item["item_id"],
        "topic_bucket": item["topic_bucket"],
        "split":        item["split"],
        "question":     question,
    }

    # --- T1: pre-explanation confidence (no solving yet) ---
    out["t1_confidence"], out["t1_raw"] = ask_confidence(
        history,
        f"Consider this question:\n\n  {question}\n\n"
        "Before answering, how confident are you that you can give a mechanistically correct, "
        "step-by-step explanation?",
    )

    # --- T2: solve + explain, then post-explanation confidence ---
    history.append({"role": "user", "content": question})
    explanation = chat(history, max_new_tokens=400, temperature=0.7, do_sample=True)
    history.append({"role": "assistant", "content": explanation})
    out["t2_explanation"] = explanation
    out["t2_confidence"], out["t2_raw"] = ask_confidence(
        history,
        "Now that you've written that explanation, how confident are you it is mechanistically "
        "correct and complete?",
    )

    # --- T3: diagnostic probe using rubric_notes (what shallow answers miss) ---
    diag = (
        "Diagnostic check. A shallow answer to this question typically misses this element:\n\n"
        f"  \"{rubric}\"\n\n"
        "Looking back at YOUR explanation above, in 2-3 sentences identify what your explanation "
        "actually says (or fails to say) about that specific mechanistic element."
    )
    history.append({"role": "user", "content": diag})
    diag_resp = chat(history, max_new_tokens=250, temperature=0.7, do_sample=True)
    history.append({"role": "assistant", "content": diag_resp})
    out["t3_diagnostic_response"] = diag_resp
    out["t3_confidence"], out["t3_raw"] = ask_confidence(
        history,
        "Given that diagnostic check, how confident are you NOW in the mechanistic correctness "
        "of your original explanation?",
    )

    # --- T4: show reference answer, final confidence ---
    out["t4_confidence"], out["t4_raw"] = ask_confidence(
        history,
        f"Here is a reference explanation that captures the key mechanistic steps:\n\n{reference}\n\n"
        "Compared to this reference, how confident are you NOW in the mechanistic correctness "
        "of YOUR original explanation?",
    )
    out["reference_shown"] = reference

    if verbose:
        traj = " -> ".join(str(out[k]) for k in ("t1_confidence","t2_confidence","t3_confidence","t4_confidence"))
        print(f"[{item['item_id']}/{item['topic_bucket']}] {question}")
        print(f"  trajectory T1->T4: {traj}")
        print(f"  T2 explanation (first 200 chars): {explanation[:200].replace(chr(10),' ')}...")
        print(f"  T3 diagnostic    (first 200 chars): {diag_resp[:200].replace(chr(10),' ')}...")
        print()
    return out


In [ ]:
# Run on the full dataset; save incrementally so a pod disconnect doesn't lose work.
from datetime import datetime
from tqdm import tqdm
import pandas as pd

sample = items  # all 80
eta_min = len(sample) * 10 / 60
print(f"Running pipeline on {len(sample)} items (eta ~{eta_min:.0f} min on H100)...\n")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = RESULTS_DIR / f"ioed_run_{ts}.json"

payload = {
    "run_metadata": {
        "timestamp":     ts,
        "model_id":      model_id,
        "dataset_path":  str(DATASET_PATH),
        "split":         "all",
        "num_items":     0,
        "sample_filter": "all 80 items (train + eval)",
    },
    "items": [],
}

for item in tqdm(sample):
    r = run_ioed_pipeline(item, verbose=False)
    payload["items"].append(r)
    payload["run_metadata"]["num_items"] = len(payload["items"])
    # Incremental save — safe against crashes / kernel disconnects.
    with out_path.open("w") as f:
        json.dump(payload, f, indent=2)
    traj = " -> ".join(str(r.get(k)) for k in ("t1_confidence","t2_confidence","t3_confidence","t4_confidence"))
    tqdm.write(f"[{r['item_id']}/{r['topic_bucket']}] T1->T4: {traj}")

print(f"\nSaved {len(payload['items'])} runs to {out_path}")

results = payload["items"]
df = pd.DataFrame([{k: r.get(k) for k in ("item_id","topic_bucket","split","t1_confidence","t2_confidence","t3_confidence","t4_confidence")} for r in results])
df["delta_T2_T1"] = df["t2_confidence"] - df["t1_confidence"]
df["delta_T3_T2"] = df["t3_confidence"] - df["t2_confidence"]
df["delta_T4_T3"] = df["t4_confidence"] - df["t3_confidence"]
df
